# Drift-Sense CPU smoke run in Google Colab

This notebook is a **reproducibility smoke test**, not a GPU training notebook. The submitted NCC matcher uses only NumPy and OpenCV. Upload a ZIP of this repository (without `data/`, `.venv/`, or `runs/`) to execute the same public generator, evaluator, and tests in a fresh Colab CPU runtime.

All output is synthetic. Do not label it as KLA/AMAT official performance.

In [ ]:
# Upload a ZIP made from the repository root.
# Example on Windows before opening Colab:
#   git archive --format=zip --output drift-sense.zip HEAD
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
!rm -rf /content/drift-sense
!mkdir -p /content/drift-sense
!unzip -q "$zip_name" -d /content/drift-sense
%cd /content/drift-sense
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt


In [ ]:
# Regression proof for the checked-in implementation.
!python -m pytest -q


In [ ]:
# Exercise the actual public generator -> manifest -> evaluator path.
# This is intentionally a small, fresh, deterministic synthetic smoke set.
!rm -rf data/colab_smoke
!python generate_dataset.py --out data/colab_smoke --n 4 --seed 31337 --arch mixed
!python evaluate.py --manifest data/colab_smoke/manifest.csv --cm-thresholds 1 5 --json-out results/colab_smoke.json


In [ ]:
# Optional tiny confidence-study smoke. Full reproducible study is 30+30
# pairs per level and takes longer:
# !python analysis/noise_sweep.py --out results/noise_sweep --levels 0 0.3 0.6 --calibration-n 30 --test-n 30 --seed 777
!python analysis/noise_sweep.py --out results/colab_noise_smoke --levels 0 --calibration-n 2 --test-n 2 --seed 777


## Evidence to retain

Download `results/colab_smoke.json`, `results/colab_noise_smoke.json`, and `results/colab_noise_smoke.svg` with the code below. Preserve the Colab runtime version, output, and the exact commit SHA with any claimed Colab result.

A Colab GPU is unnecessary for this CPU NCC method. Do not introduce Torch, LPIPS, or training code unless a new method is benchmarked against the current NCC baseline on a disjoint synthetic holdout.

In [ ]:
from google.colab import files
!git rev-parse HEAD
files.download('results/colab_smoke.json')
files.download('results/colab_noise_smoke.json')
files.download('results/colab_noise_smoke.svg')
